<a href="https://colab.research.google.com/github/ghrkarbasi/efficiency-prediction/blob/main/DEA_SBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
مدل تحلیل پوششی داده‌ها (DEA) - SBM (Slack-Based Measure)
====================================================
این اسکریپت مخصوص فایل:
    data_with_bcc_score_for_keras_nn_colab.xlsx
نوشته شده و کارآیی هر شرکت را با مدل SBM (تون، ۲۰۰۱) محاسبه می‌کند
و علاوه بر عدد کارآیی، مقادیر کاستی (اسلک) هر ورودی و خروجی را هم
در یک فایل اکسل جداگانه ذخیره می‌کند.

تفاوت SBM با BCC:
    - BCC یک مدل شعاعی (radial) است و فقط یک ضریب انقباض/انبساط یکسان
      برای همه ورودی‌ها (یا خروجی‌ها) می‌دهد.
    - SBM یک مدل غیرشعاعی (non-radial) و بدون جهت (non-oriented) است که
      مستقیماً کاستی‌های (slack) هر ورودی و هر خروجی را به‌طور جداگانه
      در نظر می‌گیرد، بنابراین معمولاً نمره‌ی کارآیی دقیق‌تر (و کمتر) از
      BCC می‌دهد.

ستون‌های فایل ورودی:
    owners equity               -> ورودی ۱
    total operational expences  -> ورودی ۲
    net profit                  -> خروجی ۱
    operational income          -> خروجی ۲
    bcc score                   -> مقدار قبلی (فقط مرجع، در محاسبه استفاده نمی‌شود)
"""

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pathlib
import os
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [ ]:
ls

 1-s2.0-S2096232020300469-main.pdf
'applied math nikukar.pdf'
 bcc_results.xlsx
 calculus/
'Colab Notebooks'/
 dataset/
 dataset2/
'data with bcc score.xlsx'
'Data work'/
 data.xlsx
'deep learninig'/
'Discrete mathematics'/
 generate_data.xlsx
'ja-had elmi.pdf'
 love/
 Machine-Learning-Tom-Mitchell_MatlabKar.com.pdf
 MachinLearning/
 mathematics-11-04873.pdf
'my CV kholaseh.pdf'
'My musice'/
'numpy quiz.png'
'ons ba quran'/
 OR/
'PDF Reader.pdf'
 projects/
'python quiz.png'
 segah.mp3
 TrustWalletBackup/
 Voic/


In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill

# ============================== settings ==============================
INPUT_FILE = "data.xlsx"
OUTPUT_FILE = "sbm_results.xlsx"

INPUT_COLS = ["owners equity", "total operational expences"]
OUTPUT_COLS = ["net profit", "operational income"]

RETURNS_TO_SCALE = "VRS"   # "VRS" ( BCC) یا "CRS" ( CCR)
# =======================================================================

In [ ]:
def sbm_efficiency(X: np.ndarray, Y: np.ndarray, vrs: bool = True):
    """
    محاسبه کارآیی SBM (غیرشعاعی، بدون جهت) برای هر شرکت (DMU) با استفاده
    از خطی‌سازی چارنز-کوپر (Tone, 2001).

    X : ماتریس ورودی‌ها به شکل (تعداد شرکت‌ها, تعداد ورودی‌ها)
    Y : ماتریس خروجی‌ها به شکل (تعداد شرکت‌ها, تعداد خروجی‌ها)
    vrs : True برای بازده متغیر به مقیاس (مثل BCC)، False برای بازده ثابت (مثل CCR)

    خروجی:
        scores   : آرایه نمرات کارآیی SBM (بین 0 و 1)
        slacks_in  : ماتریس کاستی ورودی‌ها (تعداد شرکت‌ها × تعداد ورودی‌ها)
        slacks_out : ماتریس کاستی خروجی‌ها (تعداد شرکت‌ها × تعداد خروجی‌ها)
        statuses : لیست "کارا"/"ناکارا"
    """
    n, m = X.shape
    s = Y.shape[1]

    scores = np.full(n, np.nan)
    slacks_in = np.full((n, m), np.nan)
    slacks_out = np.full((n, s), np.nan)
    statuses = [""] * n

    # متغیرها: [t, S-_1..m, S+_1..s, Lambda_1..n]
    nvar = 1 + m + s + n
    idx_t = 0
    idx_Sm = list(range(1, 1 + m))
    idx_Sp = list(range(1 + m, 1 + m + s))
    idx_L = list(range(1 + m + s, 1 + m + s + n))

    for o in range(n):
        c = np.zeros(nvar)
        c[idx_t] = 1.0
        for i in range(m):
            c[idx_Sm[i]] = -1.0 / (m * X[o, i])

        A_eq, b_eq = [], []

        # t + (1/s) * sum_r S+_r / y_ro = 1
        row = np.zeros(nvar)
        row[idx_t] = 1.0
        for r in range(s):
            row[idx_Sp[r]] = 1.0 / (s * Y[o, r])
        A_eq.append(row); b_eq.append(1.0)

        # x_io*t - sum_j Lambda_j*x_ij - S-_i = 0  (for each inputs)
        for i in range(m):
            row = np.zeros(nvar)
            row[idx_t] = X[o, i]
            for j in range(n):
                row[idx_L[j]] = -X[j, i]
            row[idx_Sm[i]] = -1.0
            A_eq.append(row); b_eq.append(0.0)

        # y_ro*t - sum_j Lambda_j*y_rj + S+_r = 0  (for each outputs)
        for r in range(s):
            row = np.zeros(nvar)
            row[idx_t] = Y[o, r]
            for j in range(n):
                row[idx_L[j]] = -Y[j, r]
            row[idx_Sp[r]] = 1.0
            A_eq.append(row); b_eq.append(0.0)

        # constraints (VRS): sum_j Lambda_j = t
        if vrs:
            row = np.zeros(nvar)
            row[idx_t] = -1.0
            for j in range(n):
                row[idx_L[j]] = 1.0
            A_eq.append(row); b_eq.append(0.0)

        bounds = [(0, None)] * nvar
        res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")

        if res.success:
            t = res.x[idx_t]
            scores[o] = res.fun
            if t > 1e-9:
                slacks_in[o, :] = np.array([res.x[i] for i in idx_Sm]) / t
                slacks_out[o, :] = np.array([res.x[i] for i in idx_Sp]) / t
            statuses[o] = "efficient" if res.fun >= 0.9999 else "non efficient"

    return scores, slacks_in, slacks_out, statuses

In [ ]:
def main():
    df = pd.read_excel(INPUT_FILE)

    X = df[INPUT_COLS].astype(float).values
    Y = df[OUTPUT_COLS].astype(float).values

    # مدل SBM نیاز به مقادیر ورودی/خروجی مثبت دارد (چون در مخرج کسر قرار می‌گیرند).
    # اگر مقدار صفر یا منفی وجود داشته باشد، آن را با یک عدد بسیار کوچک مثبت
    # جایگزین می‌کنیم تا محاسبات دچار خطای تقسیم بر صفر نشود.
    for name, arr in (("ورودی", X), ("خروجی", Y)):
        n_bad = np.sum(arr <= 0)
        if n_bad > 0:
            #print(f"هشدار: {n_bad} مقدار صفر/منفی در داده‌های {name} یافت شد؛ "
            #      f"با مقدار بسیار کوچک (epsilon) جایگزین شد.")
            arr[arr <= 0] = 1e-6

    vrs = RETURNS_TO_SCALE.upper() == "VRS"
    scores, slacks_in, slacks_out, statuses = sbm_efficiency(X, Y, vrs=vrs)

    result_df = df.copy()
    result_df["SBM Efficiency Score"] = scores
    for i, col in enumerate(INPUT_COLS):
        result_df[f"Slack ({col})"] = slacks_in[:, i]
    for r, col in enumerate(OUTPUT_COLS):
        result_df[f"Slack ({col})"] = slacks_out[:, r]
    result_df["Status"] = statuses

    result_df = result_df.sort_values("SBM Efficiency Score", ascending=False)
    result_df.to_excel(OUTPUT_FILE, index=False, sheet_name="SBM Results")

    # ----- Simple and professional formatting -----
    wb = load_workbook(OUTPUT_FILE)
    ws = wb["SBM Results"]

    header_font = Font(name="Arial", bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="305496", end_color="305496", fill_type="solid")
    body_font = Font(name="Arial")

    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center")

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.font = body_font

    for col_cells in ws.columns:
        max_len = max(len(str(c.value)) if c.value is not None else 0 for c in col_cells)
        ws.column_dimensions[col_cells[0].column_letter].width = min(max(14, max_len + 2), 35)

    ws.freeze_panes = "A2"
    wb.save(OUTPUT_FILE)

    print(f"resultes file saved: {OUTPUT_FILE}")
    print(f"efficient companies: {sum(1 for s in statuses if s == 'efficient')} from {len(statuses)}")


if __name__ == "__main__":
    main()

هشدار: 1 مقدار صفر/منفی در داده‌های خروجی یافت شد؛ با مقدار بسیار کوچک (epsilon) جایگزین شد.
resultes file saved: sbm_results.xlsx
efficient companies: 35 from 927
